# Group Knowledge Graph — NYC Foursquare Check-ins

Builds a heterogeneous knowledge graph combining **all users** in the
NYC Foursquare check-in data, with a group layer derived from
**ephemeral groups** (users who co-located at the same POI within a
1-hour window).

## Schema

**Nodes:** `USER`, `POI` (name/category/coords), `CATEGORY` (flat — one
node per unique full category string, no hierarchy), `GROUP` (one per
ephemeral-group instance)

**Edges (relation types):**
| Relation | Direction | Meaning |
|---|---|---|
| `VISITED` | USER -> POI | aggregated visit count + first/last timestamp |
| `MEMBER_OF` | USER -> GROUP | user participated in this ephemeral group |
| `OCCURRED_AT` | GROUP -> POI | where the group happened |
| `CO_ATTENDED` | USER <-> USER | co-occurred in >=1 ephemeral group (weight = # shared groups) |
| `SIMILAR_PREFERENCE` | USER <-> USER | cosine similarity of category-visit vectors, kept sparse (top-K neighbors above a threshold, not all pairs) |

Two separate user-user edge types: `CO_ATTENDED` (actual shared group
experiences) and `SIMILAR_PREFERENCE` (behavioral similarity across full
history, independent of ever having met).

In [12]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple
from sklearn.metrics.pairwise import cosine_similarity

## 0. Data paths

In [13]:
DATA_DIR = "/kaggle/input/datasets/eyamhamdi03/poi-final"

POI_PATH = f"{DATA_DIR}/poi_metadata_NYC.csv"
CHECKIN_PATH = f"{DATA_DIR}/train_NYC.csv"
print("Using:", POI_PATH, "|", CHECKIN_PATH)

Using: /kaggle/input/datasets/eyamhamdi03/poi-final/poi_metadata_NYC.csv | /kaggle/input/datasets/eyamhamdi03/poi-final/train_NYC.csv


## 1. Ephemeral group formation

Users who checked in at the **same POI within a 1-hour window** form an
ephemeral group (single-linkage time clustering per venue, kept only
when >=2 distinct users are involved).

In [14]:
"""
Data loading and ephemeral-group formation for MAC-GPR.

  Users who checked in at the SAME POI within a ONE-HOUR window are
  treated as an ephemeral group (co-location-based, non-repetitive,
  spontaneous groups -- as opposed to persistent/social groups).

We form groups per-venue: sort a venue's check-ins by time, then run a
single-linkage clustering along the time axis (chain rule: consecutive
check-ins whose gap <= window are merged into the same cluster). A
cluster only counts as a "group" if it contains >= 2 distinct users.
"""

from dataclasses import dataclass, field
from typing import List, Dict, Set, Tuple
import pandas as pd
import numpy as np


@dataclass
class EphemeralGroup:
    group_id: int
    venue_id: str
    poi_idx: int
    start_time: pd.Timestamp
    end_time: pd.Timestamp
    members: List[int]  # user_ids, unique


class POIStore:
    """Wraps poi_metadata_NYC.csv: poi_idx <-> category / coords / name."""

    def __init__(self, poi_metadata_path: str):
        df = pd.read_csv(poi_metadata_path)
        df["category"] = df["category"].fillna("Unknown")
        self.df = df.set_index("poi_idx")
        self.all_poi_idx = self.df.index.to_numpy()
        self.category_of = self.df["category"].to_dict()
        self.name_of = self.df["name"].to_dict()
        self.lat_of = self.df["latitude"].to_dict()
        self.lon_of = self.df["longitude"].to_dict()

    def category(self, poi_idx: int) -> str:
        return self.category_of.get(poi_idx, "Unknown")

    def coords(self, poi_idx: int) -> Tuple[float, float]:
        return self.lat_of.get(poi_idx, np.nan), self.lon_of.get(poi_idx, np.nan)

    def name(self, poi_idx: int) -> str:
        return self.name_of.get(poi_idx, f"POI#{poi_idx}")


def haversine_km(lat1, lon1, lat2, lon2) -> float:
    """Great-circle distance in km. Used for geographic Pareto-frontier
    expansion (Eq. 3's Euclidean distance d(p, p'), operationalized here
    as geographic distance between POIs -- the paper does not specify
    the embedding space for d, so we use physical proximity, which is
    the natural choice for POI recommendation)."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def load_checkins(test_csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(test_csv_path)
    df["utc_time"] = pd.to_datetime(df["utc_time"], utc=True)
    return df


def build_ephemeral_groups(
    checkins: pd.DataFrame, window_minutes: int = 60
) -> List[EphemeralGroup]:
    """Form ephemeral groups: same venue, single-linkage clustering in
    time with a `window_minutes` gap threshold, keeping only clusters
    with >= 2 distinct users."""
    groups: List[EphemeralGroup] = []
    gid = 0
    window = pd.Timedelta(minutes=window_minutes)

    for venue_id, venue_df in checkins.groupby("venue_id"):
        venue_df = venue_df.sort_values("utc_time")
        times = venue_df["utc_time"].tolist()
        users = venue_df["user_id"].tolist()
        poi_idx = venue_df["poi_idx"].iloc[0]

        cluster_times = [times[0]]
        cluster_users = [users[0]]

        def flush(cluster_times, cluster_users):
            nonlocal gid
            uniq_users = list(dict.fromkeys(cluster_users))  # preserve order, dedupe
            if len(uniq_users) >= 2:
                groups.append(
                    EphemeralGroup(
                        group_id=gid,
                        venue_id=venue_id,
                        poi_idx=poi_idx,
                        start_time=cluster_times[0],
                        end_time=cluster_times[-1],
                        members=uniq_users,
                    )
                )
                gid += 1

        for t, u in zip(times[1:], users[1:]):
            if t - cluster_times[-1] <= window:
                cluster_times.append(t)
                cluster_users.append(u)
            else:
                flush(cluster_times, cluster_users)
                cluster_times, cluster_users = [t], [u]
        flush(cluster_times, cluster_users)

    return groups

## 2. Knowledge graph construction

In [16]:
"""
Group Knowledge Graph construction.

Schema (heterogeneous graph, built with a NetworkX MultiDiGraph so
parallel edges of different relation types between the same two nodes
are preserved -- standard practice for KGs later consumed as (head,
relation, tail) triples, e.g. for hyperbolic KGE training):

Node types
----------
USER      user:{user_id}
POI       poi:{poi_idx}                (attrs: name, category, lat, lon)
GROUP     group:{group_id}             (attrs: venue, poi_idx, start_time,
                                         end_time, size)

Edge types (relation)
----------------------
VISITED             USER -> POI            attrs: count, first_ts, last_ts
MEMBER_OF           USER -> GROUP
OCCURRED_AT         GROUP -> POI
CO_ATTENDED         USER <-> USER (both directions) attrs: weight (# shared
                     ephemeral groups), group_ids
SIMILAR_PREFERENCE  USER <-> USER (both directions) attrs: weight (cosine
                     similarity of category-visit vectors), kept only for
                     each user's top-K most similar neighbors above a
                     threshold (kept sparse -- an all-pairs graph over
                     ~1000+ users would be dense and uninformative)
"""

from dataclasses import dataclass
from typing import Dict, List, Set, Tuple
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity



@dataclass
class KGConfig:
    similarity_top_k: int = 10
    similarity_threshold: float = 0.3


def add_poi_nodes(G: nx.MultiDiGraph, poi_store: POIStore):
    """POI structural layer"""
    for poi_idx in poi_store.all_poi_idx:
        poi_idx = int(poi_idx)
        lat, lon = poi_store.coords(poi_idx)

        G.add_node(
            f"poi:{poi_idx}",
            node_type="POI",
            name=poi_store.name(poi_idx),
            category=poi_store.category(poi_idx),
            lat=lat,
            lon=lon,
        )

def add_user_visit_layer(G: nx.MultiDiGraph, checkins: pd.DataFrame):
    """USER nodes + VISITED edges (aggregated per user-POI pair)."""
    for user_id, udf in checkins.groupby("user_id"):
        G.add_node(f"user:{user_id}", node_type="USER")

        # VISITED: aggregate (user, poi) -> count / first / last timestamp
        for poi_idx, pdf in udf.groupby("poi_idx"):
            u_node, p_node = f"user:{user_id}", f"poi:{int(poi_idx)}"
            G.add_edge(
                u_node, p_node, key="VISITED", relation="VISITED",
                count=len(pdf),
                first_ts=pdf["utc_time"].min().isoformat(),
                last_ts=pdf["utc_time"].max().isoformat(),
            )


def add_group_layer(G: nx.MultiDiGraph, groups: List[EphemeralGroup]):
    """GROUP nodes, MEMBER_OF, OCCURRED_AT, and CO_ATTENDED (user-user,
    weighted by number of shared ephemeral groups)."""
    co_attend_weight: Dict[Tuple[int, int], int] = {}
    co_attend_groups: Dict[Tuple[int, int], List[int]] = {}

    for g in groups:
        gid = f"group:{g.group_id}"
        G.add_node(
            gid, node_type="GROUP", venue=g.venue_id, poi_idx=int(g.poi_idx),
            start_time=g.start_time.isoformat(), end_time=g.end_time.isoformat(),
            size=len(g.members),
        )
        G.add_edge(gid, f"poi:{int(g.poi_idx)}", key="OCCURRED_AT", relation="OCCURRED_AT")

        for u in g.members:
            G.add_edge(f"user:{u}", gid, key="MEMBER_OF", relation="MEMBER_OF")

        members = sorted(g.members)
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                pair = (members[i], members[j])
                co_attend_weight[pair] = co_attend_weight.get(pair, 0) + 1
                co_attend_groups.setdefault(pair, []).append(g.group_id)

    for (u1, u2), w in co_attend_weight.items():
        gids = co_attend_groups[(u1, u2)]
        G.add_edge(f"user:{u1}", f"user:{u2}", key="CO_ATTENDED", relation="CO_ATTENDED",
                   weight=w, group_ids=gids)
        G.add_edge(f"user:{u2}", f"user:{u1}", key="CO_ATTENDED", relation="CO_ATTENDED",
                   weight=w, group_ids=gids)


def add_similarity_layer(
    G: nx.MultiDiGraph, checkins: pd.DataFrame, poi_store: POIStore, cfg: KGConfig
):
    """SIMILAR_PREFERENCE (user-user), based on cosine similarity of each
    user's category-visit-frequency vector, kept sparse: each user only
    connects to their top-`similarity_top_k` neighbors above
    `similarity_threshold` (an all-pairs graph over 1000+ users would be
    dense -- O(n^2) edges -- and mostly uninformative noise)."""
    checkins = checkins.copy()
    checkins["leaf_category"] = checkins["venue_category_name"].fillna("Unknown")

    users = sorted(checkins["user_id"].unique().tolist())
    categories = sorted(checkins["leaf_category"].unique().tolist())
    cat_index = {c: i for i, c in enumerate(categories)}
    user_index = {u: i for i, u in enumerate(users)}

    mat = np.zeros((len(users), len(categories)))
    for row in checkins.itertuples(index=False):
        mat[user_index[row.user_id], cat_index[row.leaf_category]] += 1
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    mat = mat / row_sums  # normalize to a preference distribution per user

    sim = cosine_similarity(mat)
    np.fill_diagonal(sim, -1)  # exclude self

    added: Set[Tuple[str, str]] = set()
    for i, u in enumerate(users):
        top_idx = np.argsort(sim[i])[::-1][: cfg.similarity_top_k]
        for j in top_idx:
            s = sim[i, j]
            if s < cfg.similarity_threshold:
                continue
            u2 = users[j]
            key = tuple(sorted((u, u2)))
            if key in added:
                continue
            added.add(key)
            G.add_edge(f"user:{u}", f"user:{u2}", key="SIMILAR_PREFERENCE",
                       relation="SIMILAR_PREFERENCE", weight=float(s))
            G.add_edge(f"user:{u2}", f"user:{u}", key="SIMILAR_PREFERENCE",
                       relation="SIMILAR_PREFERENCE", weight=float(s))

In [22]:
def build_group_knowledge_graph(
    checkins: pd.DataFrame,
    poi_store: POIStore,
    groups: List[EphemeralGroup],
    cfg: KGConfig = KGConfig(),
) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    add_poi_nodes(G, poi_store)
    add_user_visit_layer(G, checkins)
    add_group_layer(G, groups)
    add_similarity_layer(G, checkins, poi_store, cfg)
    return G

In [23]:
def graph_summary(G: nx.MultiDiGraph) -> pd.DataFrame:
    node_counts: Dict[str, int] = {}
    for _, data in G.nodes(data=True):
        node_counts[data.get("node_type", "?")] = node_counts.get(data.get("node_type", "?"), 0) + 1

    edge_counts: Dict[str, int] = {}
    for _, _, data in G.edges(data=True):
        edge_counts[data.get("relation", "?")] = edge_counts.get(data.get("relation", "?"), 0) + 1

    rows = [{"kind": "NODE", "type": k, "count": v} for k, v in sorted(node_counts.items())]
    rows += [{"kind": "EDGE", "type": k, "count": v} for k, v in sorted(edge_counts.items())]
    return pd.DataFrame(rows)


def export_graphml_safe(G: nx.MultiDiGraph, path: str):
    """GraphML only supports primitive (str/int/float/bool) attribute
    values, but a couple of our edge attrs are Python set/list
    (CO_ATTENDED's `group_ids`) for in-memory convenience. This writes a
    stringified copy so the graph can still be opened in Gephi / other
    GraphML tools; use `pickle` (see below) if you need the raw Python
    objects back losslessly."""
    H = G.copy()
    for _, _, data in H.edges(data=True):
        for k, v in list(data.items()):
            if isinstance(v, (set, list)):
                data[k] = ",".join(str(x) for x in v)
    nx.write_graphml(H, path)


def export_triples(G: nx.MultiDiGraph) -> pd.DataFrame:
    """Flatten to (head, relation, tail) triples -- the standard input
    format for KGE training pipelines (e.g. your Stage 3/4 RotH setup)."""
    rows = []
    for h, t, data in G.edges(data=True):
        rows.append({"head": h, "relation": data.get("relation", "?"), "tail": t})
    return pd.DataFrame(rows)

## 3. Build the graph on our data

In [24]:
checkins = load_checkins(CHECKIN_PATH)
poi_store = POIStore(POI_PATH)
groups = build_ephemeral_groups(checkins, window_minutes=60)
print(f"Formed {len(groups)} ephemeral groups (same POI, 1-hour co-location window)")

cfg = KGConfig(similarity_top_k=10, similarity_threshold=0.3)
G = build_group_knowledge_graph(checkins, poi_store, groups, cfg)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Formed 2460 ephemeral groups (same POI, 1-hour co-location window)
Graph: 8653 nodes, 79207 edges


In [28]:
def size_bucket(n_members: int) -> str:
    """Buckets, not exact counts: with 99 real groups, sizes 4 and 5
    each have only 1 group -- a node with 1 neighbor gives the KGE
    nothing to learn from. Buckets give each category real support."""
    if n_members == 2:
        return "pair"
    elif n_members <= 4:
        return "small_group"
    else:
        return "large_group"


def add_group_size_layer(G: nx.MultiDiGraph, groups: List[EphemeralGroup]):
    """GROUP_SIZE nodes (categorical, bucketed) + HAS_SIZE edges
    (GROUP -> GROUP_SIZE)."""
    for g in groups:
        bucket = size_bucket(len(g.members))
        size_id = f"size:{bucket}"
        if size_id not in G:
            G.add_node(size_id, node_type="GROUP_SIZE", label=bucket)
        G.add_edge(f"group:{g.group_id}", size_id, key="HAS_SIZE", relation="HAS_SIZE")

In [31]:
add_group_size_layer(G, groups)
print(f"Graph now: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph now: 8656 nodes, 81667 edges


In [30]:
graph_summary(G)

,kind,type,count
0,NODE,GROUP,2460
1,NODE,GROUP_SIZE,3
2,NODE,POI,5120
3,NODE,USER,1073
4,EDGE,CO_ATTENDED,23348
5,EDGE,HAS_SIZE,2460
6,EDGE,MEMBER_OF,6111
7,EDGE,OCCURRED_AT,2460
8,EDGE,SIMILAR_PREFERENCE,17050
9,EDGE,VISITED,30238


## 4. Inspect one group's local neighborhood

Sanity check: pull the subgraph around a single ephemeral group -- its
members, the POI they met at, and any CO_ATTENDED / SIMILAR_PREFERENCE
links to other users.

In [26]:
sample_group_node = "group:5"  # pick any group id from `groups`
neighbors = list(G.successors(sample_group_node)) + list(G.predecessors(sample_group_node))
print("Group node:", sample_group_node, G.nodes[sample_group_node])
print()
for n in neighbors:
    print(n, "->", G.nodes[n].get("node_type"), G.nodes[n])

Group node: group:5 {'node_type': 'GROUP', 'venue': '3fd66200f964a52008f11ee3', 'poi_idx': 8, 'start_time': '2012-05-12T21:14:00+00:00', 'end_time': '2012-05-12T21:47:01+00:00', 'size': 2}

poi:8 -> POI {'node_type': 'POI', 'name': 'Bohemian Hall & Beer Garden', 'category': 'Dining and Drinking > Bar > Beer Garden', 'lat': 40.77300701660423, 'lon': -73.9158862978038}
user:890 -> USER {'node_type': 'USER'}
user:785 -> USER {'node_type': 'USER'}


## 5. Export

- `triples.csv`: flattened (head, relation, tail) rows -- standard input
  for KGE training pipelines (e.g. your Stage 3/4 RotH setup).
- `.graphml`: for inspection in Gephi or similar (set/list attributes are
  stringified since GraphML only supports primitive types).
- `.gpickle`: full-fidelity Python pickle of the NetworkX graph.

In [27]:
triples = export_triples(G)
print("triples shape:", triples.shape)
print(triples["relation"].value_counts())

triples.to_csv("/kaggle/working/group_kg_triples.csv", index=False)
export_graphml_safe(G, "/kaggle/working/group_kg.graphml")
with open("/kaggle/working/group_kg.gpickle", "wb") as f:
    pickle.dump(G, f)

print("Saved: group_kg_triples.csv, group_kg.graphml, group_kg.gpickle")

triples shape: (79207, 3)
relation
VISITED               30238
CO_ATTENDED           23348
SIMILAR_PREFERENCE    17050
MEMBER_OF              6111
OCCURRED_AT            2460
Name: count, dtype: int64
Saved: group_kg_triples.csv, group_kg.graphml, group_kg.gpickle
